# S3.P3 实际秘密求交汇总
原始输入与秘密份额不读取。

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json,yaml,hashlib,subprocess
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/"AGENTS.md").is_file() and (p/"modules/m3_alignment").is_dir())
module=ROOT/"modules/m3_alignment"
config=module/"configs/secret_join_report.yaml"
cfg=yaml.safe_load(config.read_text())
print(json.dumps({"config":str(config),"config_sha256":hashlib.sha256(config.read_bytes()).hexdigest(),"seed":cfg["seeds"],"bootstrap_seed":cfg["bootstrap_seed"],"git_sha":subprocess.check_output(["git","rev-parse","HEAD"],cwd=ROOT,text=True).strip(),"utc":datetime.now(timezone.utc).isoformat()},indent=2))

{
  "config": "/Applications/\u603b\u6587\u4ef6\u5939/\u6c47\u4e30/HSBC-VFL-personal-project/modules/m3_alignment/configs/secret_join_report.yaml",
  "config_sha256": "b23e6e93214a3ce19eb64559089e1e49a5cda09563b194a86b39bded37b9802c",
  "seed": [
    101,
    102,
    103,
    104,
    105
  ],
  "bootstrap_seed": 20261013,
  "git_sha": "293c5afc7339800cb8e772f26863c1b4cd460906",
  "utc": "2026-10-13T01:45:15.518471+00:00"
}


In [2]:
from IPython.display import Markdown, display
print((module/"results/secret_join_v1/engineering.json").read_text())
display(Markdown((module/"通用秘密输出求交实验记录.md").read_text()))

{
  "step": "S3.P3",
  "status": "passed",
  "datasets": [
    "uci_bank_marketing",
    "hillstrom_email_marketing"
  ],
  "trials_per_dataset": 5,
  "participant_exit_codes": [
    [
      0,
      0
    ],
    [
      0,
      0
    ]
  ],
  "secret_sort_256bit_keys": true,
  "secret_all_pairs_oracle": true,
  "keys_or_shares_exported": false,
  "intersection_or_count_reconstructed": false,
  "frozen_owner_keys_unchanged": true,
  "shares_600_owner_only": true,
  "existing_training_consumes_this_psi": false,
  "native_rr22": false,
  "real_customer_matching_validated": false,
  "production_security": false,
  "rows_per_owner_public_padding": 64,
  "actual_source_queue": "/Applications/总文件夹/汇丰/8-9月/秘密输出求交/S3P3_queue_20261013T013838899648Z",
  "config_sha256": "20356716f6759a5211d9193ef54a802d43dda1c10e2fc11934bedd8e7c4940fc"
}



# 通用秘密输出求交实验记录

> 编写日期：2026-10-13 ｜ 版本：v1.0 ｜ 状态：实际完成；本地/远端发布验收接续
> 数据来源：results/secret_join_v1两任务JSON逐文件路径、SHA256、配置/源码/Notebook指纹
> 口径基准：所有者继续秘密输出；DR-M3-002/DR-M5-001/DR-GOV-012

目的：消除S3.P2依赖公开键全集枚举的应用限制，以成熟SPU/CHEETAH密态排序/比较处理本方任意非空唯一字符串键。SHA-256编码8个uint32完整字；不是自研密码原语，不称原生RR22接口或成熟工业Circuit-PSI部署。

两任务各顺序seed101/102/103/104/105，每方固定填充64行，私有熵决定保留/乱序，虚键本方独有。输入源仅既有冻结本方keys.json，管理员在对应VM复制；不读特征、标签或sealed。私有熵及实际编码向量留本方，每轮独立code/data/results/logs；公开代码和seed不能重建私有成员选择。原始源内容及新input在所有轮后又在本方核对哈希一致。

每轮密态拼接、按完整256-bit键排序、检测跨方相邻相等、恢复原行序，得到双方member mask和peer index（无匹配位置用0占位，必须始终与秘密mask共同使用）。四元组直接SPU.dump为两份membership.share，不reveal列表/索引/计数。独立all-pairs equality oracle只在MPC比较实际键，公开一个全正确布尔位；Python字典oracle仅用于纯合成已知答案。当前输入长度、控制布尔、计时允许披露，不能称无侧信道。

纯合成完全/零/部分重叠、乱序、不等长与Unicode已知答案MPC控制通过；重复/空键本方拒绝测试通过。两个任务十轮实际密码学求交及独立密态oracle通过，双方退出0，无cleanup错误；双通道mTLS、匿名连接拒绝，shares文件owner UID/600与跨方目录不可见、源/代码/配置/完整三cell Notebook均核验。宿主不导出实际键、向量、份额。控制布尔通过不是恶意安全证明；仍为可信管理员、半诚实不串通双VM模型。

下表耗时包括本方加载/编码、密态join、密态oracle、份额dump；每轮用双方耗时最大值，避免把两个相关参与方当十个独立重复。第一轮编译/缓存开销保留。500次bootstrap、seed20261013、95%分位范围仅为固定小规模串行重复测时的描述性条件范围，缓存/首轮非交换性不支持总体/工业性能外推。

|数据集|五轮双方最大耗时（秒）|平均耗时与描述性条件范围（秒）|
|---|---|---|
|uci_bank_marketing|5.539, 3.601, 3.448, 3.657, 3.487|3.946 [3.483, 4.744]|
|hillstrom_email_marketing|4.909, 3.832, 3.390, 3.611, 3.672|3.883 [3.512, 4.396]|

全部双方原计时、种子、统计范围和源码/来源指纹见两JSON与[timing.csv](results/secret_join_v1/timing.csv)。没有交集规模、输入真实样本数或小人群统计。

失败/修正记录：此前直接JAX CPU求值SIGILL（不等于SPU不支持）、frontend回调误用range(list)、VM重启后/tmp日志丢失均保留；本轮重新在持久目录获得known/join64/oracle64 frontend编译通过，未用直接JAX CPU结果替代MPC。准备时home路径按getent修正，lint格式修正；扩展测试首轮31通过/1失败源于测试fixture漏rows_per_owner，补齐fixture后40通过，不改已完成实际代码/配置/输入。所有负控制不在真实数据上造错。实际十轮无协议失败或重训。

S5.X1本地399测试/53回归、普通main293c5af及30文件字节核验先完成。为提高效率，在执行前记录时序调整，允许远端CI38332090151与本地此研究验证并行；CI未结束不能记通过，失败阻断本轮发布。旧准备“等全部CI”的建议有显式取代记录，不作事后回填。

达到：通用字符串键在本机双VM上的秘密求交输出功能，不需要先枚举公开全集。未达到：自然跨机构身份匹配、恶意/合谋/root抵抗、千万级规模、通信字节统计、重复自适应查询攻防、现有训练消费秘密对齐。构造记录键本来预对齐并可能来自可枚举的公域，不能据此验证自然客户隐私；256-bit压缩仍依赖哈希计算安全假设。

这次独立PSI不追认既有模型“端到端真实PSI训练完成”。后续必须先约束秘密mask/索引的消费能力、固定长度数据搬运/安全gather和masked loss，再单独训练验收；不得reveal交集以方便接入。分组正则、安全树、Criteo二元处理适配及一次真正未查看人群确认仍分别后续步骤。


两任务十轮MPC完成，交集输出不重建；小规模构造ID验证不等于真实客户、工业安全或历史训练已消费秘密对齐。